# Driving Cycle Combiner
Combine `Standard_WVUCITY.mat` and `Standard_HWFET.mat` into one cycle.

In [1]:
import os
import numpy as np
import scipy.io as scio


In [6]:
# --- Input / Output ---
WLTC_PATH = "/home/chahyunjoon/dev/FCEV-EMS/project-data/standard_driving_cycles/mat-data/Standard_WVUSUB.mat"
WVUINTER_PATH = "/home/chahyunjoon/dev/FCEV-EMS/project-data/standard_driving_cycles/mat-data/Standard_HWFET.mat"
OUTPUT_PATH = "/home/chahyunjoon/dev/FCEV-EMS/project-data/standard_driving_cycles/mat-data/Standard_WVUSUB_HWFET.mat"

# Optional: specify key names if not `speed_vector`
WVUCITY_KEY = None
HWFET_KEY = None

# Optional: insert zeros between cycles
GAP_SAMPLES = 0


In [7]:
def _first_vector_from_mat(data):
    candidates = []
    for k, v in data.items():
        if k.startswith("__"):
            continue
        if not isinstance(v, np.ndarray):
            continue
        if v.size == 0:
            continue
        if v.dtype.kind not in ("f", "i", "u"):
            continue
        candidates.append((k, v))
    if not candidates:
        raise KeyError("No numeric arrays found in .mat file")
    key, arr = max(candidates, key=lambda kv: kv[1].size)
    return key, arr


def load_speed(path, key=None):
    data = scio.loadmat(path)
    if key and key in data:
        arr = data[key]
    elif "speed_vector" in data:
        arr = data["speed_vector"]
    else:
        k, arr = _first_vector_from_mat(data)
        print(f"[warn] speed_vector not found in {os.path.basename(path)}; using '{k}'")
    speed = np.asarray(arr).squeeze()
    if speed.ndim != 1:
        speed = speed.reshape(-1)
    return speed


In [8]:
wvu = load_speed(WLTC_PATH, WVUCITY_KEY)
hwf = load_speed(WVUINTER_PATH, HWFET_KEY)

if GAP_SAMPLES > 0:
    gap = np.zeros(GAP_SAMPLES, dtype=wvu.dtype)
    combined = np.concatenate([wvu, gap, hwf])
else:
    combined = np.concatenate([wvu, hwf])

print("WVUCITY samples:", wvu.size)
print("HWFET samples:", hwf.size)
print("Combined samples:", combined.size)


WVUCITY samples: 1665
HWFET samples: 766
Combined samples: 2431


In [9]:
out = {
    "speed_vector": combined.reshape(1, -1),
    "source": np.array([os.path.basename(WLTC_PATH), os.path.basename(WVUINTER_PATH)], dtype=object),
}
scio.savemat(OUTPUT_PATH, out)
print("Saved:", OUTPUT_PATH)


Saved: /home/chahyunjoon/dev/FCEV-EMS/project-data/standard_driving_cycles/mat-data/Standard_WVUSUB_HWFET.mat
